# Практическая работа №9. Индексы и производительность в PostgreSQL
### Шаблон для выполнения в Google Colab (через `psycopg2`)

Впишите свой **номер варианта** и параметры подключения к серверу в ячейке ниже, а затем выполняйте шаги работы по порядку.

In [ ]:
!pip install -q psycopg2-binary pandas matplotlib

In [ ]:
import re
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

# === ЗАПОЛНИТЕ СВОИ ДАННЫЕ ===
VARIANT_NUM = 1                        # Ваш номер варианта (от 1 до 20)
DB_HOST     = "IP_ВАШЕГО_СЕРВЕРА"      # Выдает преподаватель
DB_PORT     = 443                      # 443 (или 5432)
DB_NAME     = f"club_var{VARIANT_NUM:02d}"
DB_USER     = f"student_{VARIANT_NUM:02d}"
DB_PASS     = f"StudyPass2026_var{VARIANT_NUM:02d}"

MY_USER_ID  = 100 + VARIANT_NUM * 17
print(f"Вариант: {VARIANT_NUM:02d} | База: {DB_NAME} | my_user_id = {MY_USER_ID}")

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASS
)
conn.autocommit = True
print("[OK] Подключение к PostgreSQL установлено!")

def run_explain(sql_query, title="EXPLAIN ANALYZE"):
    """Выполняет EXPLAIN ANALYZE, красиво выводит план и возвращает Execution Time (мс)."""
    with conn.cursor() as cur:
        cur.execute("EXPLAIN ANALYZE " + sql_query)
        rows = [r[0] for r in cur.fetchall()]
    print(f"=== {title} ===")
    exec_ms = None
    for line in rows:
        print(line)
        m = re.search(r"Execution Time:\s*([0-9.]+)\s*ms", line)
        if m:
            exec_ms = float(m.group(1))
    print("-" * 60)
    return exec_ms

---
## Шаг 1. Создание полигона `orders` на 200 000 строк

In [ ]:
setup_sql = """
DROP TABLE IF EXISTS bench_with_idx CASCADE;
DROP TABLE IF EXISTS bench_no_idx CASCADE;
DROP TABLE IF EXISTS orders CASCADE;

CREATE TABLE orders (
    order_id    SERIAL PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    order_date  TIMESTAMP NOT NULL,
    status      VARCHAR(20) NOT NULL,
    amount      NUMERIC(10, 2) NOT NULL
);

INSERT INTO orders (user_id, order_date, status, amount)
SELECT
    (floor(random() * 2000) + 1)::INTEGER,
    NOW() - (random() * 365) * INTERVAL '1 day',
    CASE
        WHEN gs % 20 = 0 THEN 'cancelled'
        WHEN gs % 10 = 1 THEN 'paid'
        ELSE 'delivered'
    END,
    ROUND((random() * 4900 + 100)::NUMERIC, 2)
FROM generate_series(1, 200000) AS gs;

ANALYZE orders;
"""

with conn.cursor() as cur:
    cur.execute(setup_sql)

df_cnt = pd.read_sql("SELECT status, COUNT(*) AS rows_cnt FROM orders GROUP BY status ORDER BY rows_cnt DESC;", conn)
display(df_cnt)

---
## Шаг 2. Точечный поиск (`Seq Scan` $\to$ `Index Scan`)
1. Замерьте время поиска по `MY_USER_ID` без индекса.
2. Создайте индекс `idx_orders_user_id`.
3. Замерьте время повторно.

In [ ]:
query_step2 = f"SELECT * FROM orders WHERE user_id = {MY_USER_ID};"

# 1. Замер ДО индекса
t_search_before = run_explain(query_step2, "Шаг 2 (ДО индекса)")

# 2. TODO: создайте индекс idx_orders_user_id на orders(user_id) и выполните ANALYZE orders;
with conn.cursor() as cur:
    cur.execute("CREATE INDEX idx_orders_user_id ON orders(user_id);")
    cur.execute("ANALYZE orders;")

# 3. Замер ПОСЛЕ индекса
t_search_after = run_explain(query_step2, "Шаг 2 (ПОСЛЕ индекса)")
print(f"Ускорение поиска: в {t_search_before / t_search_after:.1f} раз!")

---
## Шаг 3. Поиск + сортировка (устранение узла `Sort`)

In [ ]:
query_step3 = f"""
SELECT * FROM orders
WHERE user_id = {MY_USER_ID}
ORDER BY order_date DESC
LIMIT 5;
"""

# 1. Замер с обычным индексом (обратите внимание на узел Sort!)
t_sort_before = run_explain(query_step3, "Шаг 3 (С одним индексом по user_id)")

# 2. Создаем составной индекс (user_id, order_date DESC)
with conn.cursor() as cur:
    cur.execute("CREATE INDEX idx_orders_user_date ON orders(user_id, order_date DESC);")
    cur.execute("ANALYZE orders;")

# 3. Замер с составным индексом (узел Sort должен исчезнуть!)
t_sort_after = run_explain(query_step3, "Шаг 3 (С составным индексом)")

---
## Шаг 4. Цена индексов — замер скорости вставки `INSERT` (50 000 строк)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        DROP TABLE IF EXISTS bench_no_idx;
        DROP TABLE IF EXISTS bench_with_idx;

        CREATE TABLE bench_no_idx AS SELECT * FROM orders WITH NO DATA;
        CREATE TABLE bench_with_idx AS SELECT * FROM orders WITH NO DATA;

        CREATE INDEX idx_b_user   ON bench_with_idx(user_id);
        CREATE INDEX idx_b_date   ON bench_with_idx(order_date);
        CREATE INDEX idx_b_status ON bench_with_idx(status);
    """)

t_ins_no_idx   = run_explain("INSERT INTO bench_no_idx SELECT * FROM orders LIMIT 50000;", "Вставка 50k БЕЗ индексов")
t_ins_with_idx = run_explain("INSERT INTO bench_with_idx SELECT * FROM orders LIMIT 50000;", "Вставка 50k С 3 индексами")
print(f"Замедление вставки: в {t_ins_with_idx / t_ins_no_idx:.2f} раза")

# Удаляем временные таблицы после замера
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS bench_no_idx; DROP TABLE IF EXISTS bench_with_idx;")

---
## Шаг 5. Частичный индекс (`Partial Index`) и сравнение размеров на диске

In [ ]:
with conn.cursor() as cur:
    cur.execute("CREATE INDEX idx_orders_cancelled ON orders(user_id) WHERE status = 'cancelled';")
    cur.execute("ANALYZE orders;")

df_sizes = pd.read_sql("""
SELECT
    pg_size_pretty(pg_relation_size('idx_orders_user_id'))   AS full_index_size,
    pg_size_pretty(pg_relation_size('idx_orders_user_date')) AS composite_index_size,
    pg_size_pretty(pg_relation_size('idx_orders_cancelled')) AS partial_index_size;
""", conn)
display(df_sizes)

run_explain(f"SELECT * FROM orders WHERE user_id = {MY_USER_ID} AND status = 'cancelled';", "Поиск отмен с частичным индексом")

---
## Итоговый график замеров и закрытие соединения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(["Без индекса (Seq Scan)", "С индексом (Index Scan)"], [t_search_before, t_search_after], color=["#d9534f", "#5cb85c"])
axes[0].set_title("Шаг 2: Время поиска SELECT (мс)")
axes[0].set_ylabel("мс")

axes[1].bar(["Без индексов", "С 3 индексами"], [t_ins_no_idx, t_ins_with_idx], color=["#5bc0de", "#f0ad4e"])
axes[1].set_title("Шаг 4: Вставка 50 000 строк INSERT (мс)")
axes[1].set_ylabel("мс")

plt.tight_layout()
plt.show()

conn.close()
print("Соединение закрыто.")

---
## Ответы на контрольные вопросы (заполните перед сдачей)

1. **Чем `EXPLAIN` отличается от `EXPLAIN ANALYZE`?**
   *Ваш ответ:* ...
2. **Почему на таблице из 100 строк PostgreSQL игнорирует индекс и делает `Seq Scan`?**
   *Ваш ответ:* ...
3. **В каком случае нужен составной индекс `(user_id, order_date DESC)`, а в каком — частичный `WHERE status = 'cancelled'`?**
   *Ваш ответ:* ...
4. **Почему нельзя создавать индексы на каждый столбец таблицы?**
   *Ваш ответ:* ...